# ForgeAI 판단 일관성 검증 — Colab 실행 환경

**목적**: 멀티에이전트 파이프라인의 판단 일관성을 30-run(5회 × 6 stratum) 으로 측정  
**환경**: Google Colab Pro (GPU: T4/A100)  
**LLM**: Ollama + qwen2.5:7b (GPU 가속)  
**임베딩**: nomic-embed-text (Ollama)

## 실행 순서
1. GPU 런타임 설정 → `런타임 > 런타임 유형 변경 > T4 GPU`
2. Cell 1 ~ 7 순서대로 실행
3. Cell 7 완료 후 리포트·차트 자동 다운로드


## Cell 1 — GPU 확인 및 Ollama 설치

In [1]:
import subprocess, os, time, shutil

# GPU 확인
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip() or "없음 (CPU 실행 — 속도 느림)")

# Ollama 설치
print("Ollama 설치 중...")
os.system("curl -fsSL https://ollama.com/install.sh | sh")

# PATH에 Ollama 경로 추가
os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")

# Ollama 바이너리 위치 확인
ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"
print(f"Ollama 경로: {ollama_bin}")

# Ollama 서버 백그라운드 시작
subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# 서버 응답 확인
r = subprocess.run(["curl", "-s", "http://localhost:11434/api/tags"], capture_output=True, text=True)
print("Ollama 서버 상태:", "OK" if r.returncode == 0 else "FAILED")

# 이후 셀을 위해 전역 변수 저장
OLLAMA_BIN = ollama_bin


SyntaxError: unterminated string literal (detected at line 9) (2820929562.py, line 9)

## Cell 2 — 모델 다운로드 (5~10분)

In [ ]:
import subprocess, time, shutil, os

OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"

models = [
    ("qwen2.5:7b",       "LLM (채팅 에이전트)"),
    ("nomic-embed-text", "임베딩 (ChromaDB 벡터 검색)"),
]

for model, desc in models:
    print(f"  다운로드 중: {model} ({desc})...")
    t0 = time.time()
    r = subprocess.run([OLLAMA_BIN, "pull", model], capture_output=True, text=True)
    elapsed = time.time() - t0
    status = "✓" if r.returncode == 0 else "✗"
    print(f"  {status} {model} ({elapsed:.0f}초)")
    if r.returncode != 0:
        print(f"    오류: {r.stderr[-300:]}")

r = subprocess.run([OLLAMA_BIN, "list"], capture_output=True, text=True)
print("
설치된 모델:
", r.stdout)


## Cell 3 — 저장소 클론 및 의존성 설치

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/enjoylonelines/ForgeAI.git'
REPO_DIR = '/content/ForgeAI'

if not os.path.exists(REPO_DIR):
    print('저장소 클론 중...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('이미 존재 — pull 실행')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'작업 디렉터리: {os.getcwd()}')

print('\nPython 패키지 설치 중...')
pkgs = [
    'langchain>=0.3.0', 'langchain-ollama>=0.2.0', 'langchain-chroma>=0.1.0',
    'langgraph>=0.2.0', 'chromadb>=0.5.0', 'pydantic>=2.7.0',
    'pandas>=2.2.0', 'ucimlrepo>=0.0.7', 'langfuse>=2.0.0',
    'pypdf>=4.2.0', 'python-dotenv>=1.0.1',
    'xgboost>=2.0.0', 'scikit-learn>=1.4.0',
    'matplotlib>=3.8.0', 'numpy>=1.26.0', 'nest_asyncio',
]
subprocess.run(['pip', 'install', '-q'] + pkgs, check=True)
print('패키지 설치 완료 ✓')


## Cell 4 — 환경 변수 설정

In [ ]:
import os

os.environ.update({
    'OLLAMA_BASE_URL':       'http://localhost:11434',
    'OLLAMA_CHAT_MODEL':     'qwen2.5:7b',
    'OLLAMA_EMBED_MODEL':    'nomic-embed-text',
    'OLLAMA_TIMEOUT_SECONDS':'300',
    'OLLAMA_NUM_CTX':        '2048',
    'OLLAMA_NUM_PREDICT':    '512',
    'CHROMA_PERSIST_DIR':    '/content/ForgeAI/data/chroma',
    'DECISION_LOG_PATH':     '/content/ForgeAI/logs/decisions.jsonl',
    'LANGFUSE_ENABLED':      'false',
    'LOG_LEVEL':             'WARNING',
})

os.makedirs('/content/ForgeAI/logs', exist_ok=True)
os.makedirs('/content/ForgeAI/docs', exist_ok=True)

r = subprocess.run(['curl', '-s', 'http://localhost:11434/api/tags'], capture_output=True, text=True)
print('Ollama 연결:', 'OK' if r.returncode == 0 else 'FAILED')
print('환경 설정 완료 ✓')


## Cell 5 — SOP 문서 인덱싱 (ChromaDB 구축)

In [ ]:
import sys, asyncio, shutil
from pathlib import Path
sys.path.insert(0, '/content/ForgeAI')

import nest_asyncio
nest_asyncio.apply()

from core.config import get_settings
get_settings.cache_clear()

async def reindex():
    from rag.ingestion import ingest_document
    sop_dir   = Path('/content/ForgeAI/data/sop_docs')
    chroma_dir = Path('/content/ForgeAI/data/chroma')
    if chroma_dir.exists():
        shutil.rmtree(chroma_dir)

    files = sorted(sop_dir.glob('*.md'))
    print(f'SOP 문서 {len(files)}건 인덱싱 중...')
    last = None
    for f in files:
        last = await ingest_document(
            file_bytes=f.read_bytes(),
            filename=f.name,
            content_type='text/markdown',
        )
        print(f'  ✓ {f.name}  ({last.chunk_count} chunks)')
    print(f'\n인덱싱 완료 — 컬렉션 총 청크: {last.collection_total if last else 0}')

asyncio.run(reindex())


## Cell 6 — 30-run 판단 일관성 검증 실행

> **예상 소요**: T4 GPU 기준 약 30~60분  
> **출력물**: `docs/consistency_report.md`, `docs/consistency_distribution.png`


In [ ]:
import sys, asyncio, time
from pathlib import Path
sys.path.insert(0, '/content/ForgeAI')

import nest_asyncio
nest_asyncio.apply()

from core.config import get_settings
get_settings.cache_clear()
from core import langchain_client
langchain_client.get_chat_llm.cache_clear()
langchain_client.get_embeddings.cache_clear()

from scripts.consistency_protocol import (
    load_stratified_samples, run_once,
    build_report, save_distribution_chart,
    consistency_rate, first_divergence_stage,
)
from pipeline.forge_pipeline import ForgePipeline

N_RUNS    = 5
N_SAMPLES = 1
TARGET    = 0.99
OUT_MD    = Path('/content/ForgeAI/docs/consistency_report.md')
OUT_PNG   = Path('/content/ForgeAI/docs/consistency_distribution.png')

print(f'층화 샘플 로드 중 (stratum당 {N_SAMPLES}개)...')
samples = load_stratified_samples(n_per_stratum=N_SAMPLES)
print(f'샘플 {len(samples)}건 → {N_RUNS}회 × {len(samples)}건 = {N_RUNS * len(samples)}회 실행\n')

async def run_all():
    pipeline = ForgePipeline()
    all_results = []
    t_total = time.monotonic()

    for idx, (stratum, log) in enumerate(samples, 1):
        print(f'[{idx:02d}/{len(samples)}] {stratum} {log.equipment_id} × {N_RUNS}회 ...', end=' ', flush=True)
        t0 = time.monotonic()
        runs = [await run_once(pipeline, log) for _ in range(N_RUNS)]

        a_vals  = [r['has_anomaly']    for r in runs]
        rc_vals = [r['recommendation'] for r in runs if r['recommendation'] is not None]
        ro_vals = [r['route']          for r in runs if r['route']          is not None]
        gs_vals = [r['grounding_score'] for r in runs if r['grounding_score'] is not None]

        result = {
            'stratum': stratum, 'equipment_id': log.equipment_id,
            'runs': runs, 'runs_rec': rc_vals, 'runs_route': ro_vals,
            'anomaly_consistency':  consistency_rate(a_vals),
            'rec_consistency':      consistency_rate(rc_vals) if rc_vals else None,
            'route_consistency':    consistency_rate(ro_vals) if ro_vals else None,
            'grounding_scores': gs_vals,
            'divergence_stage': first_divergence_stage([r['stages'] for r in runs]),
        }
        all_results.append(result)

        elapsed = time.monotonic() - t0
        fmt = lambda v: f'{v*100:.0f}%' if v is not None else 'N/A'
        measurable = [v for v in [result['anomaly_consistency'], result['rec_consistency'], result['route_consistency']] if v is not None]
        status = '✅' if (measurable and min(measurable) >= TARGET) else '❌'
        print(f"{status} anomaly={fmt(result['anomaly_consistency'])} rec={fmt(result['rec_consistency'])} route={fmt(result['route_consistency'])} ({elapsed:.0f}s)")

    elapsed_total = time.monotonic() - t_total
    report = build_report(all_results, N_RUNS, elapsed_total, TARGET)
    print('\n' + '='*60 + '\n' + report)

    OUT_MD.write_text(report, encoding='utf-8')
    print(f'리포트 저장: {OUT_MD}')
    save_distribution_chart(all_results, OUT_PNG)
    return all_results

results = asyncio.run(run_all())


## Cell 7 — 결과 다운로드

In [ ]:
from google.colab import files
from IPython.display import Image, display
from pathlib import Path

OUT_MD  = Path('/content/ForgeAI/docs/consistency_report.md')
OUT_PNG = Path('/content/ForgeAI/docs/consistency_distribution.png')

if OUT_PNG.exists():
    print('판정 분포 차트:')
    display(Image(filename=str(OUT_PNG)))

for f in [OUT_MD, OUT_PNG]:
    if f.exists():
        files.download(str(f))
        print(f'다운로드: {f.name}')
    else:
        print(f'파일 없음: {f}')
